# Stage 6c — Recall timing of post-recall citations (post hoc; DEVIATIONS.md D8)

The primary rule (Stage 6) counts a citation as post-recall when the citing 510(k) was **received after
the predicate's recall was public**. This notebook places every citation from a non-recalled device
cleared after the predicate's recall began into timing categories, and reports sensitivity analyses.

| Category | Rule | Meaning |
|---|---|---|
| **A** public before submission | public < received | counted by the primary rule |
| **B** initiated, not yet public | initiated < received <= public | recall existed; an outside sponsor could not have seen it |
| **C** submitted before recall | received <= initiated < cleared | recall began during FDA review |
| **U** uncertain | a date missing, or received = initiated | |

Dates come from the frozen snapshot: recall initiation (Stage 5, earliest valid), recall public date
(earliest recall-database posting or Enforcement Report listing), 510(k) receipt and decision.
Writes `data/citation_timing.csv` and `data/timing_summary.json`. Network: none. Run after Stage 6.

In [1]:
import gzip, json
import numpy as np
import pandas as pd

D = lambda s: pd.to_datetime(s, errors="coerce")
corpus = pd.read_csv("data/corpus.csv", dtype=str)
year = dict(zip(corpus["k_number"], pd.to_numeric(corpus["decision_year"], errors="coerce")))
applicant = dict(zip(corpus["k_number"], corpus["applicant"]))
edges = pd.read_csv("data/predicate_edges.csv", dtype=str).drop_duplicates(["predicate_knumber", "device_knumber"])
recalled = pd.read_csv("data/recalled_nodes.csv", dtype=str, keep_default_na=False)
EXPECTED = json.load(open("data/expected_values.json"))
assert EXPECTED.get("post_recall_rule") == "received_after_public", "run Stage 6 (2026-10 version) first"

k510 = json.load(gzip.open("snapshot/510k_raw.json.gz", "rt"))
received = {r["k_number"]: D(r.get("date_received")) for r in k510}
decision = {r["k_number"]: D(r.get("decision_date")) for r in k510}
init = {k: D(v) for k, v in zip(recalled["k_number"], recalled["event_date_initiated"])}
wclass = dict(zip(recalled["k_number"], recalled["worst_class"]))
recalled_set = set(recalled["k_number"]) & set(corpus["k_number"])

# public dates by source, per recalled device (earliest across its linked recalls)
report = {e.get("recall_number"): D(e.get("report_date"))
          for e in json.load(gzip.open("snapshot/enforcement_raw.json.gz", "rt"))}
posted_min, report_min, firms = {}, {}, {}
for r in json.load(gzip.open("snapshot/recall_raw.json.gz", "rt")):
    po, rp = D(r.get("event_date_posted")), report.get(r.get("product_res_number"), pd.NaT)
    for k in r.get("k_numbers") or []:
        k = str(k).strip().upper()
        if k not in recalled_set: continue
        firms.setdefault(k, set()).add(r.get("recalling_firm") or "")
        if pd.notna(po) and (k not in posted_min or po < posted_min[k]): posted_min[k] = po
        if pd.notna(rp) and (k not in report_min or rp < report_min[k]): report_min[k] = rp
public = {k: min([d for d in (posted_min.get(k), report_min.get(k)) if d is not None and pd.notna(d)], default=pd.NaT)
          for k in recalled_set}
stage6 = pd.read_csv("data/recall_public_dates.csv", dtype=str)
assert all(public[k] == pd.Timestamp(v) for k, v in zip(stage6["k_number"], stage6["recall_public_date"])), \
    "public dates differ from Stage 6's data/recall_public_dates.csv"
print(f"CHECKPOINT  recalled devices {len(recalled_set)} | with public date {sum(pd.notna(v) for v in public.values())}")

CHECKPOINT  recalled devices 1691 | with public date 1689


### Classify every citation cleared after the predicate's recall began

In [2]:
def norm(s):
    s = (s or "").lower()
    for t in [",", ".", " inc", " llc", " corporation", " corp", " ltd", " gmbh", " co", " usa", " us", " ag", " l.p", " lp", " sa"]:
        s = s.replace(t, " ")
    return " ".join(s.split()[:2])

def same_firm(a, b):   # crude first-word name match; a lower bound (no corporate-family map yet)
    nb = norm(applicant.get(b))
    cands = {norm(applicant.get(a))} | {norm(f) for f in firms.get(a, set()) if f}
    return bool(nb) and any(c and nb.split()[0] == c.split()[0] for c in cands)

def classify(ri, pub, rec, dec, lag=0):
    if pd.isna(rec) or pd.isna(dec): return "U"
    if pd.notna(pub) and pub + pd.Timedelta(days=lag) < rec: return "A"
    if rec < ri: return "C"
    if rec == ri or pd.isna(pub): return "U"
    return "B"

rows = []
for a, b in zip(edges["predicate_knumber"], edges["device_knumber"]):
    if a not in recalled_set or b in recalled_set or b not in year: continue
    ri, dec = init.get(a, pd.NaT), decision.get(b, pd.NaT)
    if pd.isna(ri) or pd.isna(dec) or dec <= ri: continue          # cleared before the recall began
    pub, rec = public.get(a, pd.NaT), received.get(b, pd.NaT)
    rows.append(dict(predicate=a, citing=b, recall_class_worst=wclass.get(a), recall_initiated=ri,
                     recall_public=pub, recall_posted=posted_min.get(a, pd.NaT), recall_report=report_min.get(a, pd.NaT),
                     b_received=rec, b_decision=dec, same_firm=same_firm(a, b),
                     year_rule=bool(pd.notna(year[b]) and year[b] > ri.year),
                     category=classify(ri, pub, rec, dec), cat_lag90=classify(ri, pub, rec, dec, 90),
                     cat_lag180=classify(ri, pub, rec, dec, 180),
                     cat_posted_only=classify(ri, posted_min.get(a, pd.NaT), rec, dec),
                     cat_report_only=classify(ri, report_min.get(a, pd.NaT), rec, dec)))
C = pd.DataFrame(rows)
C["year_gap"] = C["b_decision"].dt.year - C["recall_initiated"].dt.year
C["review_days"] = (C["b_decision"] - C["b_received"]).dt.days
C["days_init_to_public"] = (C["recall_public"] - C["recall_initiated"]).dt.days
C["days_init_to_receipt"] = (C["b_received"] - C["recall_initiated"]).dt.days
C.sort_values(["predicate", "b_received"]).to_csv("data/citation_timing.csv", index=False, date_format="%Y-%m-%d")

triple = lambda d: [int(d["predicate"].nunique()), int(len(d)), int(d["citing"].nunique())]
tA = triple(C[C["category"] == "A"])
tY = triple(C[C["year_rule"]])
print(f"CHECKPOINT  citations cleared after recall initiation {len(C)} | category A {tA} (Stage 6 primary "
      f"{[EXPECTED['persistent_predicates'], EXPECTED['post_recall_citations'], EXPECTED['downstream_devices']]})")
print(f"CHECKPOINT  year-rule citations {tY} (Stage 6 sens_year_rule {EXPECTED['sens_year_rule']})")
assert tA == [EXPECTED["persistent_predicates"], EXPECTED["post_recall_citations"], EXPECTED["downstream_devices"]]
assert tY == EXPECTED["sens_year_rule"]

CHECKPOINT  citations cleared after recall initiation 2837 | category A [893, 2553, 1709] (Stage 6 primary [893, 2553, 1709])
CHECKPOINT  year-rule citations [910, 2643, 1762] (Stage 6 sens_year_rule [910, 2643, 1762])


### Tables

In [3]:
CATS = ["A", "B", "C", "U"]
def tab(col, d):
    n = len(d); vc = d[col].value_counts()
    return {c: [int(vc.get(c, 0)), round(100 * vc.get(c, 0) / n, 1) if n else 0.0] for c in CATS} | {"n": int(n)}

res = {"universe_cleared_after_initiation": triple(C), "primary": tab("category", C)}
for col in ["cat_lag90", "cat_lag180", "cat_posted_only", "cat_report_only"]:
    res[col] = tab(col, C)
res["triple_A"] = tA
res["triple_A_lag90"] = triple(C[C["cat_lag90"] == "A"])
res["triple_A_lag180"] = triple(C[C["cat_lag180"] == "A"])
res["triple_submitted_after_initiation"] = triple(C[C["category"].isin(["A", "B"])])
res["triple_C"] = triple(C[C["category"] == "C"])
res["year_rule_citations_by_category"] = tab("category", C[C["year_rule"]])
res["category_A_not_in_year_rule"] = int(((C["category"] == "A") & ~C["year_rule"]).sum())
res["by_year_gap"] = {str(g): tab("category", d) for g, d in C.groupby(C["year_gap"].clip(upper=4))}
res["by_class"] = {str(g): tab("category", d) for g, d in C.groupby(C["recall_class_worst"].replace("", "n/a"))}
res["by_same_firm_crude"] = {str(g): tab("category", d) for g, d in C.groupby("same_firm")}
res["same_firm_share_of_A_crude"] = round(100 * C.loc[C["category"] == "A", "same_firm"].mean(), 1)
res["timing"] = {
    "review_days_median": float(C["review_days"].median()),
    "review_days_iqr": [float(C["review_days"].quantile(.25)), float(C["review_days"].quantile(.75))],
    "init_to_public_days_median": float(C["days_init_to_public"].median()),
    "init_to_public_days_iqr": [float(C["days_init_to_public"].quantile(.25)), float(C["days_init_to_public"].quantile(.75))],
    "init_to_receipt_years_median_A": round(float(C.loc[C["category"] == "A", "days_init_to_receipt"].median()) / 365.25, 2),
}
g = C.groupby("predicate")["category"].agg(set)
res["devices"] = {"any_A": int(g.apply(lambda s: "A" in s).sum()),
                  "only_C_or_U": int(g.apply(lambda s: s <= {"C", "U"}).sum()),
                  "only_B_C_or_U": int(g.apply(lambda s: "A" not in s).sum())}
# 'first cited only after recall': Stage 6 values, plus the old year-rule devices re-checked on dates
res["first_after_recall"] = {"primary": EXPECTED["first_after_recall"],
                             "year_rule": EXPECTED["first_after_recall_year_rule"]}
json.dump(res, open("data/timing_summary.json", "w"), indent=1)

print(pd.DataFrame({k: {c: f"{v[c][0]} ({v[c][1]}%)" for c in CATS} | {"n": v["n"]}
                    for k, v in [("primary", res["primary"]), ("lag 90 d", res["cat_lag90"]), ("lag 180 d", res["cat_lag180"]),
                                 ("posted only", res["cat_posted_only"]), ("report only", res["cat_report_only"]),
                                 ("year-rule citations", res["year_rule_citations_by_category"])]}).to_string())
print(f"CHECKPOINT  A triple {tA} | lag90 {res['triple_A_lag90']} | lag180 {res['triple_A_lag180']} | "
      f"submitted after initiation {res['triple_submitted_after_initiation']} | C {res['triple_C']}")
print(f"CHECKPOINT  wrote data/citation_timing.csv ({len(C)} rows) and data/timing_summary.json")

        primary      lag 90 d     lag 180 d   posted only   report only year-rule citations
A  2553 (90.0%)  2435 (85.8%)  2323 (81.9%)  2124 (74.9%)  1755 (61.9%)        2510 (95.0%)
B    119 (4.2%)    237 (8.4%)   349 (12.3%)    133 (4.7%)    219 (7.7%)           76 (2.9%)
C    159 (5.6%)    159 (5.6%)    159 (5.6%)    159 (5.6%)    159 (5.6%)           51 (1.9%)
U      6 (0.2%)      6 (0.2%)      6 (0.2%)   421 (14.8%)   704 (24.8%)            6 (0.2%)
n          2837          2837          2837          2837          2837                2643
CHECKPOINT  A triple [893, 2553, 1709] | lag90 [859, 2435, 1631] | lag180 [834, 2323, 1553] | submitted after initiation [914, 2672, 1799] | C [143, 159, 143]
CHECKPOINT  wrote data/citation_timing.csv (2837 rows) and data/timing_summary.json
